In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.models as models

In [ ]:
##############################
# Dataset for Test Data
##############################
class XRayDataset(Dataset):
    def __init__(self, csv_file, img_dir, transform=None):
        self.df = pd.read_csv(csv_file)
        self.img_dir = img_dir
        self.transform = transform
        # Expect columns: image_name, Atelectasis, Effusion, Infiltration, Normal

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.img_dir, row['Image Index'])
        img = Image.open(img_path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        labels = row[['Atelectasis','Effusion','Infiltration','Normal']].values.astype(np.float32)
        labels = torch.tensor(labels)
        return img, labels, img_path


In [ ]:

##############################
# Multi-label ResNet101 Model
##############################
class MultiLabelResNet101(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = models.resnet101(pretrained=True)
        num_ftrs = self.backbone.fc.in_features
        self.backbone.fc = nn.Identity()
        # four classifier heads
        self.head_atelectasis  = nn.Linear(num_ftrs, 1)
        self.head_effusion     = nn.Linear(num_ftrs, 1)
        self.head_infiltration = nn.Linear(num_ftrs, 1)
        self.head_normal       = nn.Linear(num_ftrs, 1)

    def forward(self, x):
        feats = self.backbone(x)
        a = self.head_atelectasis(feats)
        e = self.head_effusion(feats)
        i = self.head_infiltration(feats)
        n = self.head_normal(feats)
        return torch.cat([a, e, i, n], dim=1)  # shape (B,4)


In [ ]:

##############################
# GradCAM Helper Class
##############################
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        self.handles = []
        self._register_hooks()

    def _register_hooks(self):
        def forward_hook(mod, inp, out):
            self.activations = out.detach()
        def backward_hook(mod, grad_in, grad_out):
            self.gradients = grad_out[0].detach()
        self.handles.append(self.target_layer.register_forward_hook(forward_hook))
        self.handles.append(self.target_layer.register_backward_hook(backward_hook))

    def generate_cam(self, x, target_class):
        out = self.model(x)
        self.model.zero_grad()
        out[0, target_class].backward(retain_graph=True)
        weights = self.gradients.mean(dim=(2,3), keepdim=True)
        cam = (weights * self.activations).sum(dim=1).squeeze()
        cam = torch.relu(cam)
        cam -= cam.min()
        if cam.max() != 0:
            cam /= cam.max()
        cam = cam.cpu().numpy()
        cam = cv2.resize(cam, (x.shape[3], x.shape[2]))
        return cam

    def remove_hooks(self):
        for h in self.handles:
            h.remove()

In [ ]:
##############################
# Visualization Utilities
##############################
def overlay_heatmap(img, heatmap, alpha=0.5, colormap=cv2.COLORMAP_JET):
    hm_uint8 = np.uint8(255 * heatmap)
    hm_color = cv2.applyColorMap(hm_uint8, colormap)
    hm_color = cv2.cvtColor(hm_color, cv2.COLOR_BGR2RGB)
    return cv2.addWeighted(img, 1-alpha, hm_color, alpha, 0)

def annotate_image(img, heatmap, label, threshold=0.3):
    ys, xs = np.where(heatmap > threshold * heatmap.max())
    if len(xs)>0 and len(ys)>0:
        x0, y0 = int(xs.mean()), int(ys.mean())
        # cv2.putText(img, label, (x0,y0), cv2.FONT_HERSHEY_SIMPLEX,
        #             0.8, (255,255,255), 2, cv2.LINE_AA)
        cv2.putText(img, label, (x0, y0), cv2.FONT_HERSHEY_SIMPLEX,
                    0.5, (255, 255, 255), 1, cv2.LINE_AA)
    return img

In [ ]:
##############################
# Main: Run
##############################
def main():
    # test_csv    = 'path/to/test_labels.csv'
    # test_dir    = 'path/to/test_images'
    # ckpt_path   = 'path/to/model.pth'

    prob_thresh = 0.3

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    transform = transforms.Compose([
        transforms.Resize((224,224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
    ])

    dataset = XRayDataset(test_csv, test_dir, transform)
    loader  = DataLoader(dataset, batch_size=1, shuffle=False, num_workers=0)

    model = MultiLabelResNet101().to(device)
    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    model.eval()

    target_layer = model.backbone.layer4[-1]
    methods = {'GradCAM'   : GradCAM(model, target_layer)}

    labels = ['Atelectasis','Effusion','Infiltration','Normal']

    for idx, (img_t, lbls, paths) in enumerate(loader):
        orig = Image.open(paths[0]).convert('RGB').resize((224,224))
        orig_np = np.array(orig)

        img_t = img_t.to(device)
        out  = model(img_t)
        probs = torch.sigmoid(out).squeeze().cpu().detach().numpy()

        overlays = {}
        for name, m in methods.items():
            cams = []
            indiv = {}
            for i, lab in enumerate(labels):
                if probs[i] >= prob_thresh:
                    cam = m.generate_cam(img_t, target_class=i)
                    indiv[lab] = cam.copy()
                    cams.append(cam)
                else:
                    indiv[lab] = np.zeros((224,224), dtype=np.float32)
            comp = np.maximum.reduce(cams) if cams else np.zeros((224,224), dtype=np.float32)
            ov = overlay_heatmap(orig_np, comp, alpha=0.5)
            for lab in labels:
                if probs[labels.index(lab)] >= prob_thresh:
                    ov = annotate_image(ov, indiv[lab], lab, threshold=0.3)
            overlays[name] = ov

        # Plot
        fig, axs = plt.subplots(1, 1, figsize=(5,5))
        for ax, (nm, ov) in zip(axs, overlays.items()):
            ax.imshow(ov)
            ax.set_title(nm)
            ax.axis('off')
        fn = os.path.basename(paths[0])
        fig.suptitle(f"{fn}\nTrue: {lbls.squeeze().numpy()} | Prob: {np.around(probs,2)}", fontsize=14)
        plt.tight_layout(rect=[0,0,1,0.92])
        plt.show()

    for m in methods.values():
        m.remove_hooks()


if __name__ == '__main__':
    main()